## BMLL Notebook Educational Series
### Trade Markout Analysis

BMLL collects exchange data from over 100 exchanges across the globe. With this core market data,
we rebuild Level 1 (top of book), Level 2 (or Market by Price), and Level 3 (or Market by Order) views.
We also provide lit and dark trades classified by aggressor side.

In [ ]:
dbutils.widgets.text("lid", "469878")
dbutils.widgets.text("date", "2024-02-07")

lid = dbutils.widgets.get("lid")
date = dbutils.widgets.get("date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
    # No dedicated sample views for these tables — fall back to production views
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"

## Understanding Trade Markouts

A **trade markout** measures the price movement of a security relative to the execution price
following a trade execution. This metric helps assess execution quality and the impact of trades
on subsequent price movements.

### Markout Concept
For each trade, we measure the mid-price (average of best bid and ask) at various points in time:
- **Pre-trade markout** (negative time): Price movement BEFORE the trade (what was the market opportunity?)
- **Post-trade markout** (positive time): Price movement AFTER the trade (was there price drift?)

The markout is calculated as:
```
Markout(t) = (Mid_Price(t) - Execution_Price) / Execution_Price × Direction × 10000
```
Where:
- **Mid_Price(t)** = Average of best bid and ask at time t relative to trade
- **Execution_Price** = The price at which the trade executed
- **Direction** = +1 for buy (bid aggressor), -1 for sell (ask aggressor)
- **×10000** = Converts to basis points for readability

### Interpretation
- **Positive markout**: Price moved favorably after trade (good execution quality)
- **Negative markout**: Price moved unfavorably (poor execution quality or post-trade drift)
- **Pre-trade markout**: Indicates whether traders are hitting stale quotes or fresh prices

### Time Horizons
We measure markout at multiple time intervals (1s, 5s, 10s, 30s, 60s) to understand:
- **1-5 seconds**: Immediate price impact and microstructure effects
- **10-30 seconds**: Short-term mean reversion or momentum
- **60 seconds**: Longer-term price drift

This notebook illustrates the methods to compute and analyze trade markouts in Databricks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from datetime import datetime, timedelta

## Data Loading Functions

We need two types of data to compute markouts:
1. **Trades**: Individual trade records with timestamp, price, size, and aggressor side
2. **Best Bid/Offer (BBO)**: Level 1 market data showing the best bid and ask prices at each timestamp

The functions below handle platform-specific data access (notebook API, Snowflake SQL, Databricks SQL).
They normalize column names and data types so the markout computation is identical across platforms.

In [ ]:
def get_trades(lid, date):
    """
    Databricks implementation: Fetch all trades from TRADES table using Spark SQL.

    Same signature and return schema as notebook version - only data source differs.
    Uses Spark for distributed computation on large datasets.
    Aggressor side is decoded from numeric values (1=BID, 2=ASK).

    Parameters:
        lid (str): BMLL Listing ID
        date (str): Date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Columns [trade_timestamp, price, execution_size, bmll_trade_type, aggressor_side]
    """
    sql = f"""
        SELECT
            TRADE_TIMESTAMP AS trade_timestamp,
            PRICE AS price,
            SIZE AS execution_size,
            BMLL_TRADE_TYPE AS bmll_trade_type,
            CASE AGGRESSOR_SIDE
                WHEN 1 THEN 'BID'
                WHEN 2 THEN 'ASK'
                ELSE NULL
            END AS aggressor_side
        FROM {TRADES_TBL}
        WHERE BMLL_OBJECT_ID = {lid}
          AND TRADE_DATE = '{date}'
          AND HOUR(TRADE_TIMESTAMP) BETWEEN 13 AND 22
        ORDER BY TRADE_TIMESTAMP
    """
    return spark.sql(sql).toPandas()


def get_bbo(lid, date):
    """
    Databricks implementation: Fetch Level 1 (BBO) data from L2_DATA_EQUITY table using Spark SQL.

    Same signature and return schema as notebook version.
    Uses Spark for efficient distributed computation.

    Parameters:
        lid (str): BMLL Listing ID
        date (str): Date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Columns [event_timestamp, best_bid_price, best_bid_size, best_ask_price, best_ask_size]
    """
    sql = f"""
        SELECT
            EVENT_TIMESTAMP AS event_timestamp,
            BID_PRICE1 AS best_bid_price,
            BID_QUANTITY1 AS best_bid_size,
            ASK_PRICE1 AS best_ask_price,
            ASK_QUANTITY1 AS best_ask_size
        FROM `market_data`.`vw_l2_data_equity`
        WHERE BMLL_OBJECT_ID = {lid}
          AND TRADE_DATE = '{date}'
          AND HOUR(EVENT_TIMESTAMP) BETWEEN 13 AND 22
        ORDER BY EVENT_TIMESTAMP
    """
    return spark.sql(sql).toPandas()

## Computing Trade Markouts

### Algorithm Overview

The markout computation follows these steps:

1. **Data Preparation**:
   - Convert timestamps to UTC for consistent time handling
   - Filter trades to regular US trading hours (9:50 AM - 4:00 PM EST)
   - Keep only lit and OTC trades with valid aggressor side (BID or ASK)
   - Map aggressor side to direction: BID→+1, ASK→-1

2. **BBO Alignment**:
   - Convert BBO timestamps to UTC
   - Calculate mid-price = (best_ask_price + best_bid_price) / 2
   - Use merge_asof to align each trade with its contemporary BBO

3. **Markout Calculation at Multiple Horizons**:
   - For each post-trade interval (1s, 5s, 10s, 30s, 60s):
     - Calculate future timestamp = trade_timestamp + interval
     - Find the BBO at that future time
     - Compute markout = (future_mid - trade_price) / trade_price × direction × 10000
   - Same for pre-trade intervals (looking back)

4. **Output**:
   - Return complete trade dataframe with all markout columns
   - Each trade has 10 markout values (5 post-trade + 5 pre-trade)

In [ ]:
def _compute_markouts_from_frames(at, bbo):
    """
    Compute trade markouts at multiple post-trade and pre-trade time horizons.

    This is the core markout calculation engine. It takes raw trade and BBO data and produces
    a dataframe where each row is a trade with markout values calculated at 5 post-trade intervals
    (1s, 5s, 10s, 30s, 60s) and 5 pre-trade intervals.

    Parameters:
        at (pd.DataFrame): Trade data with columns [trade_timestamp, price, execution_size,
                          bmll_trade_type, aggressor_side]
        bbo (pd.DataFrame): Level 1 BBO data with columns [event_timestamp, best_bid_price,
                           best_bid_size, best_ask_price, best_ask_size]

    Returns:
        pd.DataFrame: Input trades plus markout columns:
                     - mo_1s, mo_5s, mo_10s, mo_30s, mo_60s: Post-trade markouts (seconds)
                     - mo_m1s, mo_m5s, mo_m10s, mo_m30s, mo_m60s: Pre-trade markouts (seconds)
                     - direction: +1 for buys, -1 for sells
                     - mid: Mid-price at trade timestamp
                     - size: Trade notional value (price × execution_size)

    Note:
        - Filters to regular US trading hours only (9:50 AM - 4:00 PM EST)
        - Includes only LIT and OTC trades with valid aggressor side
        - Uses merge_asof for efficient time-based matching of BBO quotes
    """
    # Step 1: Timestamp normalization - Convert to UTC for consistent time handling
    at['trade_timestamp'] = pd.to_datetime(at['trade_timestamp'], utc=True)
    at['trade_timestamp_est'] = at['trade_timestamp'].dt.tz_convert('US/Eastern')

    # Step 2: Filter to regular US trading hours (9:50 AM - 4:00 PM EST)
    # Excludes pre-market (opens at 9:30, but we start at 9:50) and after-hours trading
    # which have different liquidity and microstructure characteristics
    at = at[
        ((at['trade_timestamp_est'].dt.hour > 9) |
         ((at['trade_timestamp_est'].dt.hour == 9) & (at['trade_timestamp_est'].dt.minute >= 50))) &
        (at['trade_timestamp_est'].dt.hour < 16)
    ]

    # Step 3: Filter trade types - Keep only lit and OTC trades
    # Remove dark trades since we need observable BBO for markout calculation
    at.drop(columns=['trade_timestamp_est'], inplace=True)
    at = at[at['bmll_trade_type'].isin(['LIT', 'OTC'])]

    # Step 4: Filter by aggressor side - Keep only valid BID/ASK
    # This ensures we can calculate directional markouts
    at = at[at['aggressor_side'].isin(['BID', 'ASK'])]

    # Step 5: Encode direction - BID buys (+1) move price up favorably, ASK sells (-1) move price down favorably
    at['direction'] = at['aggressor_side'].map({'BID': 1, 'ASK': -1})

    # Step 6: Prepare BBO data - Calculate mid-price and keep only needed columns
    bbo['event_timestamp'] = pd.to_datetime(bbo['event_timestamp'], utc=True)
    bbo['mid'] = (bbo['best_ask_price'] + bbo['best_bid_price']) / 2
    bbo = bbo[['event_timestamp', 'best_bid_size', 'best_bid_price', 'best_ask_price', 'best_ask_size', 'mid']]

    # Step 7: Sort trades by timestamp for merge_asof (requires sorted keys)
    at['trade_timestamp'] = pd.to_datetime(at['trade_timestamp'])
    at = at.sort_values(by='trade_timestamp')

    # Step 8: Align trades with their contemporary BBO (the BBO at trade time)
    at = pd.merge_asof(at, bbo, left_on='trade_timestamp', right_on='event_timestamp')
    at['mid_trade'] = at['price'] == at['mid']

    # Step 9: Calculate POST-trade markouts at 5 time intervals
    # For each interval, find the BBO at (trade_time + interval) and calculate markout
    TIME_INTERVALS = [1, 5, 10, 30, 60]  # seconds
    for interval in TIME_INTERVALS:
        col_prefix = f"mo_{interval}s"
        # Calculate future timestamp: trade_time + interval
        at[f'mo_ts_{interval}s'] = at['trade_timestamp'] + pd.Timedelta(seconds=interval)
        # Find the BBO at that future time
        at = pd.merge_asof(
            at,
            bbo,
            left_on=f'mo_ts_{interval}s',
            right_on='event_timestamp',
            suffixes=("", f"_{col_prefix}")
        )
        # Compute markout = (future_mid - trade_price) / trade_price × direction × 10000 bps
        at[f'{col_prefix}'] = 10000 * (at[f'mid_{col_prefix}'] - at['price']) / at['price'] * at['direction']

    # Step 10: Calculate PRE-trade markouts at 5 time intervals
    # Same as post-trade but looking backwards (what was the market before the trade?)
    for interval in TIME_INTERVALS:
        col_prefix = f"mo_m{interval}s"
        # Calculate past timestamp: trade_time - interval
        at[f'mo_ts_m{interval}s'] = at['trade_timestamp'] - pd.Timedelta(seconds=interval)
        # Find the BBO at that past time
        at = pd.merge_asof(
            at,
            bbo,
            left_on=f'mo_ts_m{interval}s',
            right_on='event_timestamp',
            suffixes=("", f"_{col_prefix}")
        )
        # Compute markout at past time
        at[f'{col_prefix}'] = 10000 * (at[f'mid_{col_prefix}'] - at['price']) / at['price'] * at['direction']

    # Step 11: Calculate notional value for aggregations
    at['size'] = at['price'] * at['execution_size']

    return at

### Size-Bucketed Analysis

The second analysis function groups trades by execution size to understand whether markout
varies with trade size. Market impact is typically larger for bigger orders, so we expect
post-trade markouts to become more negative (unfavorable) as size increases.

**Size Buckets** (shares):
- **<= 100**: Small retail-sized orders
- **100-500**: Medium institutional orders
- **500-1000**: Large block trades
- **> 1000**: Very large institutional blocks

For each size bucket and trade type combination, we compute size-weighted averages of markout.
This reveals whether execution quality (measured by markout) depends on order size.

In [ ]:
def _compute_markouts_mid_from_frames(at, bbo):
    """
    Compute trade markouts AND aggregate by size bucket and trade type.

    This function extends the basic markout computation by also grouping results by:
    - Trade size bucket (retail vs institutional sizes)
    - Trade type (lit vs OTC)
    - Whether trade occurred at the mid-price

    The output is aggregated at the bucket level with size-weighted averages of markout.
    This is useful for analyzing whether execution quality varies with order size.

    Parameters:
        at (pd.DataFrame): Trade data with columns [trade_timestamp, price, execution_size,
                          bmll_trade_type, aggressor_side]
        bbo (pd.DataFrame): Level 1 BBO data with columns [event_timestamp, best_bid_price,
                           best_bid_size, best_ask_price, best_ask_size]

    Returns:
        pd.DataFrame: Aggregated by [mid_trade, bmll_trade_type, size_bucket] with columns:
                     - size_sum_size_mo_*s: Size-weighted sum of markouts (post-trade)
                     - size_sumprod_size_mo_*s: Sum of (markout × notional_value) (post-trade)
                     - size_sum_size_mo_m*s: Size-weighted sum of markouts (pre-trade)
                     - size_sumprod_size_mo_m*s: Sum of (markout × notional_value) (pre-trade)

    Note:
        - Includes early morning trades (9:40 AM start vs 9:50 AM for basic analysis)
        - Filters to regular US trading hours only (9:40 AM - 4:00 PM EST)
        - Size weighting accounts for notional value differences across buckets
    """
    # Step 1: Timestamp normalization
    at['trade_timestamp'] = pd.to_datetime(at['trade_timestamp'], utc=True)
    at['trade_timestamp_est'] = at['trade_timestamp'].dt.tz_convert('US/Eastern')

    # Step 2: Filter to regular US trading hours (9:40 AM - 4:00 PM EST)
    # Note: This starts 10 minutes earlier than basic analysis to capture more data
    at = at[
        ((at['trade_timestamp_est'].dt.hour > 9) |
         ((at['trade_timestamp_est'].dt.hour == 9) & (at['trade_timestamp_est'].dt.minute >= 40))) &
        (at['trade_timestamp_est'].dt.hour < 16)
    ]
    at.drop(columns=['trade_timestamp_est'], inplace=True)

    # Step 3: Filter trade types and aggressor side
    at = at[at['bmll_trade_type'].isin(['LIT', 'OTC'])]
    at = at[at['aggressor_side'].isin(['BID', 'ASK'])]

    # Step 4: Encode direction
    at['direction'] = at['aggressor_side'].map({'BID': 1, 'ASK': -1})

    # Step 5: Classify trades by execution size
    # These buckets represent different market participant profiles:
    # - <=100: Likely retail via market makers or retail brokers
    # - 100-500: Small institutional allocations
    # - 500-1000: Medium institutional blocks
    # - >1000: Large institution allocations or block trades
    SIZE_BUCKETS = [0, 100, 500, 1000, float('inf')]
    SIZE_LABELS = ['<= 100', '100-500', '500-1000', '> 1000']
    at['size_bucket'] = pd.cut(at['execution_size'], bins=SIZE_BUCKETS, labels=SIZE_LABELS, right=False)

    # Step 6: Prepare BBO data
    bbo['event_timestamp'] = pd.to_datetime(bbo['event_timestamp'], utc=True)
    bbo['mid'] = (bbo['best_ask_price'] + bbo['best_bid_price']) / 2
    bbo = bbo[['event_timestamp', 'best_bid_size', 'best_bid_price', 'best_ask_price', 'best_ask_size', 'mid']]

    # Step 7: Sort and align trades with contemporary BBO
    at.sort_values('trade_timestamp', inplace=True)
    at = pd.merge_asof(at, bbo, left_on="trade_timestamp", right_on="event_timestamp")
    at['mid_trade'] = at['price'] == at['mid']

    # Step 8: Calculate post-trade markouts
    TIME_INTERVALS = [1, 5, 10, 30, 60]
    for interval in TIME_INTERVALS:
        col_prefix = f"mo_{interval}s"
        at[f'mo_ts_{interval}s'] = at['trade_timestamp'] + pd.Timedelta(seconds=interval)
        at = pd.merge_asof(
            at, bbo, left_on=f"mo_ts_{interval}s", right_on="event_timestamp",
            suffixes=("", f"_{col_prefix}")
        )
        at[f'mo_{interval}s'] = (
            10000 * (at[f'mid_{col_prefix}'] - at['price']) / at['price'] * at['direction']
        )

    # Step 9: Calculate pre-trade markouts
    for interval in TIME_INTERVALS:
        col_prefix = f"mo_m{interval}s"
        at[f'mo_ts_m{interval}s'] = at['trade_timestamp'] - pd.Timedelta(seconds=interval)
        at = pd.merge_asof(
            at, bbo, left_on=f'mo_ts_m{interval}s', right_on='event_timestamp',
            suffixes=("", f"_{col_prefix}")
        )
        at[f'mo_m{interval}s'] = (
            10000 * (at[f'mid_{col_prefix}'] - at['price']) / at['price'] * at['direction']
        )

    # Step 10: Calculate notional value for size weighting
    at['size'] = at['price'] * at['execution_size']

    # Step 11: Aggregate by size bucket, trade type, and mid-trade indicator
    # For each group, compute size-weighted averages of markout
    # This shows whether execution quality varies with order size
    group_cols = ['mid_trade', 'bmll_trade_type', 'size_bucket']
    agg_dict = {}

    def add_agg(field):
        # Create two aggregation metrics:
        # 1. size_sum_size_*: Sum of markout values (weighted by notional)
        # 2. size_sumprod_size_*: Sum of (markout × notional) for computing weighted average
        agg_dict[f'size_sum_size_{field}'] = (field, lambda x: at.loc[x.index, 'size'].sum())
        agg_dict[f'size_sumprod_size_{field}'] = (field, lambda x: (x * at.loc[x.index, 'size']).sum())

    # Build aggregation dictionaries for all markout columns
    for interval in TIME_INTERVALS:
        add_agg(f'mo_{interval}s')
        add_agg(f'mo_m{interval}s')

    # Step 12: Execute aggregation and return results
    result = at.groupby(group_cols).agg(**agg_dict).reset_index()
    return result.dropna()

## Example: Single Security, Single Day

Let's compute markouts for a specific security (listing ID) on a given date.
We'll look at one trading day and see the full distribution of markout values
across all trades, then visualize how markouts evolve over the trading day.

### Average Markouts Across Time Horizons

Now let's compute the average markout at each time interval to see the typical
price movement pattern. We expect:
- **Negative pre-trade markouts** (mo_m1s, mo_m5s): Market was moving before the trade
- **Positive post-trade markouts** (mo_1s, mo_5s): Good execution if positive, bad if negative
- **Evolving pattern**: Whether the market continues in the same direction or mean-reverts

### Visualization: Markout Profile

This chart shows the "markout profile" - how the mid-price moves relative to
the execution price over time. The shape of this profile reveals important
aspects of market microstructure:

- **V-shaped profile**: Price moved to an extreme before the trade, then reverted
- **J-shaped profile (post-trade)**: Execution was on stale quotes (drift continued)
- **Symmetric profile**: Market was efficient - trade occurred at fair value

## Example: Multiple Securities, Multiple Days

For larger-scale analysis, we can compute markouts across many securities and dates.
This reveals patterns in execution quality across the market.

In [ ]:
# For Databricks: resolve BMLL_OBJECT_ID from vw_trades using AAPL/XNAS as the example security.
# The widget default (469878) is an internal BMLL Lab ID; marketplace vw_trades uses BMLL_OBJECT_ID.
_db_lid = spark.sql(f"""SELECT DISTINCT `BMLL_OBJECT_ID` FROM {TRADES_TBL} WHERE `TICKER` = 'AAPL' AND `MIC` = 'XNAS' AND `TRADE_DATE` = '{date}' LIMIT 1""").toPandas()
if not _db_lid.empty:
    lid = str(int(_db_lid.iloc[0]["BMLL_OBJECT_ID"]))
else:
    raise ValueError(f"Could not find BMLL_OBJECT_ID for AAPL/XNAS on {date} in vw_trades — no data available")

In [ ]:
# Example: Analyze a list of listings
all_us_prim_lids = [lid]  # In practice, this would be a larger list of listing IDs


def get_markouts(lid, date):
    """
    Wrapper function: Compute trade markouts for a listing on a specific date.

    Orchestrates the full markout computation pipeline: fetches data, computes markouts,
    and handles errors gracefully. This is the main entry point for single-date analysis.

    Parameters:
        lid (str): BMLL Listing ID
        date (str): Date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Complete trade data with markout columns [mo_1s, mo_5s, ..., mo_m60s]
                     plus listing ID. Returns None if computation fails.

    Note:
        Catches exceptions during computation and logs errors to allow batch processing
        to continue even if individual dates/listings have issues.
    """
    try:
        # Fetch raw data
        at = get_trades(lid, date)
        bbo = get_bbo(lid, date)

        # Compute markouts using shared algorithm
        at = _compute_markouts_from_frames(at, bbo)

        # Add listing ID for traceability
        at['lid'] = lid

        return at

    except Exception as e:
        print(f"[ERROR] Failed to compute markouts for {lid} on {date}: {e}")
        return None


In [ ]:
start_date_str = "2023-10-01"
end_date_str = "2023-12-31"

# # %% [markdown]
# ### Batch Processing: Multiple Securities and Dates
#
# For larger-scale analysis, we process markouts across many listing IDs and dates.
# This reveals patterns in execution quality across time and across securities.
#
# The batch processing approach:
# 1. Create a list of (listing_id, date) pairs for all combinations of securities and dates
# 2. Use distributed computation (SparkHelper for notebook, list comprehension for SQL platforms)
# 3. Combine results into a single dataframe
# 4. Aggregate to compute average markouts across the full dataset

In [ ]:
# Define analysis period
start_date_str = date
end_date_str = date

# Convert to datetime and create business day range
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
date_list = pd.bdate_range(start_date, end_date)

In [ ]:
# Create all combinations of (listing_id, date)
# In practice, all_us_prim_lids would contain many listing IDs
args = [(lid, date.strftime("%Y-%m-%d")) for lid in all_us_prim_lids for date in date_list]

### Parallel Computation

The notebook environment uses SparkHelper for efficient distributed computation.
Snowflake/Databricks use list comprehension for SQL-based data access.

## Databricks: Markout Analysis — Two Approaches

On Databricks, loading large L2/BBO DataFrames to the driver via `.toPandas()` causes OOM.
Instead, both approaches below keep all computation in Spark and only collect the small
aggregated result.

**Approach 1 — Pre-computed (recommended):** `trades_plus` ships with
`PreTradeMid{N}msAtPrimary` and `PostTradeMid{N}msAtPrimary` columns pre-calculated at
12 time intervals (1 ms to 60 s). Markouts are derived in Spark with `F.col` arithmetic.
No L2 data loaded at all.

**Approach 2 — Custom Spark (AS-OF join):** Joins `trades` and `l2_data_equity` in
distributed Spark SQL using `QUALIFY ROW_NUMBER()` for the AS-OF lookup at each interval.
Use this when you need non-standard intervals, a custom mid definition, or cross-venue BBO.

In [ ]:
from pyspark.sql import functions as F

# Example security — resolved from TRADES so BMLL_OBJECT_ID is consistent across both approaches
_ticker = "AAPL"
_mic    = "XNAS"

### Approach 1: Pre-computed markouts from `trades_plus`

In [ ]:
# Millisecond intervals available in trades_plus
_PRE_MS  = [60000, 30000, 5000, 1000, 500, 100, 50, 25, 10, 5, 2, 1]
_POST_MS = [1, 2, 5, 10, 25, 50, 100, 500, 1000, 5000, 30000, 60000]

_pre_cols  = ", ".join(f"PRE_TRADE_MID_{ms}MS_AT_PRIMARY"  for ms in _PRE_MS)
_post_cols = ", ".join(f"POST_TRADE_MID_{ms}MS_AT_PRIMARY" for ms in _POST_MS)

_tp = spark.sql(f"""
    SELECT
        TRADE_NOTIONAL_USD,
        CASE AGGRESSOR_SIDE WHEN 1 THEN 1 WHEN 2 THEN -1 ELSE NULL END AS direction,
        (BEST_BID_PRICE_AT_PRIMARY + BEST_ASK_PRICE_AT_PRIMARY) / 2     AS mid_at_trade,
        {_pre_cols},
        {_post_cols}
    FROM {TRADES_PLUS_TBL}
    WHERE TICKER     = '{_ticker}'
      AND MIC        = '{_mic}'
      AND TRADE_DATE = '{date}'
      AND BMLL_TRADE_TYPE IN ('LIT', 'OTC')
      AND AGGRESSOR_SIDE IN (1, 2)
""")

# Compute markouts in Spark: bps = 10000 × (mid_offset − mid_at_trade) / mid_at_trade × direction
for _ms in _PRE_MS:
    _c = f"PRE_TRADE_MID_{_ms}MS_AT_PRIMARY"
    _tp = _tp.withColumn(f"mo_m{_ms}ms",
        F.when(F.col(_c).isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(_c) - F.col("mid_at_trade")) / F.col("mid_at_trade") * F.col("direction")))

for _ms in _POST_MS:
    _c = f"POST_TRADE_MID_{_ms}MS_AT_PRIMARY"
    _tp = _tp.withColumn(f"mo_{_ms}ms",
        F.when(F.col(_c).isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(_c) - F.col("mid_at_trade")) / F.col("mid_at_trade") * F.col("direction")))

# Aggregate: notional-weighted average entirely in Spark; only the small summary hits the driver
_mo_cols_1 = [f"mo_m{ms}ms" for ms in _PRE_MS] + [f"mo_{ms}ms" for ms in _POST_MS]
_agg1 = [(F.sum(F.col(c) * F.col("TRADE_NOTIONAL_USD")) / F.sum("TRADE_NOTIONAL_USD")).alias(c)
         for c in _mo_cols_1]
result_precomp = _tp.agg(*_agg1).toPandas().melt(var_name="interval", value_name="markout_bps")
result_precomp["approach"] = "pre-computed (trades_plus)"
result_precomp

### Approach 2: Custom markouts via Spark AS-OF join (`trades` + `l2_data_equity`)

In [ ]:
# Coarser interval set matching the notebook/Snowflake variants for direct comparison
_INTERVALS_S = {"mo_m60s": -60, "mo_m30s": -30, "mo_m5s": -5, "mo_m1s": -1,
                "mo_1s":     1,  "mo_5s":   5,  "mo_30s": 30, "mo_60s": 60}

# Key optimisation: aggregate BBO to 1-second buckets in Spark before any join.
# Collapses millions of tick-level rows to ~23K rows (6.5h × 3600s).
# The resulting table is small enough to broadcast, making all joins equi-joins with no shuffle.

# Collect the ~23K-row aggregated BBO to the driver once, then re-distribute.
# This avoids re-scanning L2_DATA_EQUITY for each of the 9 broadcast joins
# (serverless doesn't support .cache()/.persist(), so this is the workaround).
_bbo_1s_pd = spark.sql(f"""SELECT DATE_TRUNC('SECOND', `EVENT_TIMESTAMP`) AS `ts_sec`, AVG((`BID_PRICE1` + `ASK_PRICE1`) / 2) AS `mid` FROM `market_data`.`vw_l2_data_equity` WHERE `TICKER` = '{_ticker}' AND `MIC` = '{_mic}' AND `TRADE_DATE` = '{date}' AND HOUR(`EVENT_TIMESTAMP`) BETWEEN 13 AND 22 GROUP BY DATE_TRUNC('SECOND', `EVENT_TIMESTAMP`)""").toPandas()  # ~23K rows — safe to collect once
_bbo_1s = spark.createDataFrame(_bbo_1s_pd)  # re-distribute for broadcast joins

_trades_s = spark.sql(f"""SELECT DATE_TRUNC('SECOND', `TRADE_TIMESTAMP`) AS `ts_sec`, `TRADE_TIMESTAMP`, `PRICE`, `SIZE`, CASE `AGGRESSOR_SIDE` WHEN 1 THEN 1 WHEN 2 THEN -1 END AS `direction` FROM {TRADES_TBL} WHERE `TICKER` = '{_ticker}' AND `MIC` = '{_mic}' AND `TRADE_DATE` = '{date}' AND `BMLL_TRADE_TYPE` IN ('LIT', 'OTC') AND `AGGRESSOR_SIDE` IN (1, 2) AND HOUR(`TRADE_TIMESTAMP`) BETWEEN 13 AND 22""")

# Broadcast-join mid at each offset: O(N_trades) per interval, no shuffle
_enriched = _trades_s
for _col, _off_s in {**{"mo_0s": 0}, **_INTERVALS_S}.items():
    _alias = "mid_at_trade" if _col == "mo_0s" else f"{_col}_mid"
    _bbo_off = F.broadcast(
        _bbo_1s.select(
            (F.col("ts_sec") + F.expr(f"INTERVAL {_off_s} SECONDS")).alias("_join_ts"),
            F.col("mid").alias(_alias)
        )
    )
    _enriched = _enriched.join(_bbo_off, _enriched.ts_sec == _bbo_off["_join_ts"], "left").drop("_join_ts")

# Compute markouts and aggregate entirely in Spark
for _col in _INTERVALS_S:
    _enriched = _enriched.withColumn(_col,
        F.when(F.col(f"{_col}_mid").isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(f"{_col}_mid") - F.col("mid_at_trade"))
                     / F.col("mid_at_trade") * F.col("direction")))

_notional = F.col("PRICE") * F.col("SIZE")
_agg2 = [(F.sum(F.col(c) * _notional) / F.sum(_notional)).alias(c) for c in _INTERVALS_S]
result_custom = _enriched.agg(*_agg2).toPandas().melt(var_name="interval", value_name="markout_bps")
result_custom["approach"] = "custom (trades + L2)"
result_custom

### Comparison: Pre-computed vs Custom

In [ ]:
import plotly.graph_objects as go

# Map ms labels → seconds for x-axis ordering alongside the custom approach
_label_to_s = {f"mo_m{ms}ms": -ms / 1000 for ms in reversed(_PRE_MS)}
_label_to_s.update({f"mo_{ms}ms": ms / 1000 for ms in _POST_MS})
_label_to_s.update({c: s for c, s in _INTERVALS_S.items()})

_combined = pd.concat([result_precomp, result_custom], ignore_index=True)
_combined["t_s"] = _combined["interval"].map(_label_to_s)
_combined = _combined.dropna(subset=["t_s", "markout_bps"]).sort_values("t_s")

fig = go.Figure()
for _approach, _grp in _combined.groupby("approach"):
    fig.add_trace(go.Scatter(
        x=_grp["t_s"], y=_grp["markout_bps"],
        mode="lines+markers", name=_approach,
    ))

fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Fair Value")
fig.add_vline(x=0, line_dash="dot",  line_color="gray")
fig.update_layout(
    title=f"Markout Profile: {_ticker}/{_mic} on {date} — Pre-computed vs Custom",
    xaxis_title="Time relative to trade (seconds)",
    yaxis_title="Notional-weighted markout (bps)",
    hovermode="x unified", template="plotly_white",
)
fig.show()

### Part A — Breakdown by Aggressor Side

Using the Approach 1 `trades_plus` data (`_tp`), compute notional-weighted average markouts
separately for BID-aggressed (AGGRESSOR_SIDE=1) and ASK-aggressed (AGGRESSOR_SIDE=2) trades.

In [ ]:
# Re-query _tp with aggressor side retained so we can group by it
_tp_side = spark.sql(f"""
    SELECT
        TRADE_NOTIONAL_USD,
        AGGRESSOR_SIDE,
        CASE AGGRESSOR_SIDE WHEN 1 THEN 1 WHEN 2 THEN -1 ELSE NULL END AS direction,
        (BEST_BID_PRICE_AT_PRIMARY + BEST_ASK_PRICE_AT_PRIMARY) / 2     AS mid_at_trade,
        {_pre_cols},
        {_post_cols}
    FROM {TRADES_PLUS_TBL}
    WHERE TICKER     = '{_ticker}'
      AND MIC        = '{_mic}'
      AND TRADE_DATE = '{date}'
      AND BMLL_TRADE_TYPE IN ('LIT', 'OTC')
      AND AGGRESSOR_SIDE IN (1, 2)
""")

for _ms in _PRE_MS:
    _c = f"PRE_TRADE_MID_{_ms}MS_AT_PRIMARY"
    _tp_side = _tp_side.withColumn(f"mo_m{_ms}ms",
        F.when(F.col(_c).isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(_c) - F.col("mid_at_trade")) / F.col("mid_at_trade") * F.col("direction")))

for _ms in _POST_MS:
    _c = f"POST_TRADE_MID_{_ms}MS_AT_PRIMARY"
    _tp_side = _tp_side.withColumn(f"mo_{_ms}ms",
        F.when(F.col(_c).isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(_c) - F.col("mid_at_trade")) / F.col("mid_at_trade") * F.col("direction")))

_agg_side = [(F.sum(F.col(c) * F.col("TRADE_NOTIONAL_USD")) / F.sum("TRADE_NOTIONAL_USD")).alias(c)
             for c in _mo_cols_1]
result_side = (_tp_side
               .groupBy("AGGRESSOR_SIDE")
               .agg(*_agg_side)
               .toPandas()
               .melt(id_vars=["AGGRESSOR_SIDE"], var_name="interval", value_name="markout_bps"))
result_side["side_label"] = result_side["AGGRESSOR_SIDE"].map({1: "BID-aggressed (buy)", 2: "ASK-aggressed (sell)"})
result_side["t_s"] = result_side["interval"].map(_label_to_s)
result_side = result_side.dropna(subset=["t_s", "markout_bps"]).sort_values("t_s")

fig_side = go.Figure()
for _label, _grp in result_side.groupby("side_label"):
    fig_side.add_trace(go.Scatter(
        x=_grp["t_s"], y=_grp["markout_bps"],
        mode="lines+markers", name=_label,
    ))

fig_side.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Fair Value")
fig_side.add_vline(x=0, line_dash="dot", line_color="gray")
fig_side.update_layout(
    title=f"Markout Profile by Aggressor Side: {_ticker}/{_mic} on {date}",
    xaxis_title="Time relative to trade (seconds)",
    yaxis_title="Notional-weighted markout (bps)",
    hovermode="x unified", template="plotly_white",
)
fig_side.show()

### Part B — Breakdown by Notional Size Bucket

Bucket trades by `TRADE_NOTIONAL_USD` into retail / small / medium / large and compare markout curves.

In [ ]:
_tp_bucket = spark.sql(f"""
    SELECT
        TRADE_NOTIONAL_USD,
        CASE AGGRESSOR_SIDE WHEN 1 THEN 1 WHEN 2 THEN -1 ELSE NULL END AS direction,
        (BEST_BID_PRICE_AT_PRIMARY + BEST_ASK_PRICE_AT_PRIMARY) / 2     AS mid_at_trade,
        {_pre_cols},
        {_post_cols}
    FROM {TRADES_PLUS_TBL}
    WHERE TICKER     = '{_ticker}'
      AND MIC        = '{_mic}'
      AND TRADE_DATE = '{date}'
      AND BMLL_TRADE_TYPE IN ('LIT', 'OTC')
      AND AGGRESSOR_SIDE IN (1, 2)
""")

for _ms in _PRE_MS:
    _c = f"PRE_TRADE_MID_{_ms}MS_AT_PRIMARY"
    _tp_bucket = _tp_bucket.withColumn(f"mo_m{_ms}ms",
        F.when(F.col(_c).isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(_c) - F.col("mid_at_trade")) / F.col("mid_at_trade") * F.col("direction")))

for _ms in _POST_MS:
    _c = f"POST_TRADE_MID_{_ms}MS_AT_PRIMARY"
    _tp_bucket = _tp_bucket.withColumn(f"mo_{_ms}ms",
        F.when(F.col(_c).isNotNull() & F.col("mid_at_trade").isNotNull(),
               10000 * (F.col(_c) - F.col("mid_at_trade")) / F.col("mid_at_trade") * F.col("direction")))

_tp_bucket = _tp_bucket.withColumn(
    "size_bucket",
    F.when(F.col("TRADE_NOTIONAL_USD") < 10_000,    "Retail (<$10K)")
     .when(F.col("TRADE_NOTIONAL_USD") < 100_000,   "Small ($10K-$100K)")
     .when(F.col("TRADE_NOTIONAL_USD") < 1_000_000, "Medium ($100K-$1M)")
     .otherwise(                                      "Large (>=$1M)")
)

_agg_bucket = [(F.sum(F.col(c) * F.col("TRADE_NOTIONAL_USD")) / F.sum("TRADE_NOTIONAL_USD")).alias(c)
               for c in _mo_cols_1]
result_bucket = (_tp_bucket
                 .groupBy("size_bucket")
                 .agg(*_agg_bucket)
                 .toPandas()
                 .melt(id_vars=["size_bucket"], var_name="interval", value_name="markout_bps"))
result_bucket["t_s"] = result_bucket["interval"].map(_label_to_s)
result_bucket = result_bucket.dropna(subset=["t_s", "markout_bps"]).sort_values("t_s")

_bucket_order = ["Retail (<$10K)", "Small ($10K-$100K)", "Medium ($100K-$1M)", "Large (>=$1M)"]
fig_bucket = go.Figure()
for _bucket in _bucket_order:
    _grp = result_bucket[result_bucket["size_bucket"] == _bucket]
    if _grp.empty:
        continue
    fig_bucket.add_trace(go.Scatter(
        x=_grp["t_s"], y=_grp["markout_bps"],
        mode="lines+markers", name=_bucket,
    ))

fig_bucket.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Fair Value")
fig_bucket.add_vline(x=0, line_dash="dot", line_color="gray")
fig_bucket.update_layout(
    title=f"Markout Profile by Notional Size Bucket: {_ticker}/{_mic} on {date}",
    xaxis_title="Time relative to trade (seconds)",
    yaxis_title="Notional-weighted markout (bps)",
    hovermode="x unified", template="plotly_white",
)
fig_bucket.show()

## Size-Bucketed Analysis: Multiple Securities and Dates

The second analysis function `get_markouts_mid` computes markouts aggregated by:
- **Trade size bucket** (retail vs institutional sizes)
- **Trade type** (lit vs OTC)
- **Mid-price indicator** (whether trade occurred exactly at mid-price)

This reveals whether execution quality varies based on order size and trade characteristics.
Institutional investors care about this analysis since larger orders often experience
worse markouts due to market impact.

In [ ]:
def get_markouts_mid(sip_lid, date):
    """
    Wrapper: Compute size-bucketed markouts for a listing on a specific date.

    Extends basic markout analysis by aggregating results by:
    - Execution size bucket (retail vs institutional)
    - Trade type (lit vs OTC)
    - Whether trade occurred at mid-price

    Useful for understanding whether execution quality depends on trade size.

    Parameters:
        sip_lid (str): BMLL Listing ID
        date (str): Date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Aggregated by [mid_trade, bmll_trade_type, size_bucket] with
                     size-weighted average markouts. Returns None if computation fails.
    """
    try:
        at = get_trades(sip_lid, date)
        bbo = get_bbo(sip_lid, date)
        return _compute_markouts_mid_from_frames(at, bbo)
    except Exception as e:
        print(f"[ERROR] Failed for {sip_lid} on {date}: {e}")
        return None


In [ ]:
all_us_sql = f"""SELECT DISTINCT `INSTRUMENT_ID` FROM {REFERENCE_TBL} WHERE `MIC` = '@SIP' AND `IS_ALIVE` = TRUE"""
all_us_iids = spark.sql(all_us_sql).toPandas()["INSTRUMENT_ID"].tolist()
if all_us_iids:
    iid_list = ", ".join(str(i) for i in all_us_iids)
    all_us_prim_sql = f"""
        SELECT LISTING_ID
        FROM {REFERENCE_TBL}
        WHERE INSTRUMENT_ID IN ({iid_list})
          AND IS_ALIVE = TRUE
          AND IS_PRIMARY = TRUE
    """
    all_us_prim_lids = spark.sql(all_us_prim_sql).toPandas()["LISTING_ID"].tolist()
else:
    all_us_prim_lids = []

In [ ]:
start_date_str = "2023-10-01"
end_date_str = "2023-12-31"

# Parse string dates to datetime objects
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

# Generate list of datetime objects for each day in the range (inclusive)
date_list = [start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)]

# If you want strings instead of datetime objects for dates
date_list_str = [date.strftime("%Y-%m-%d") for date in date_list]

# Create a list of tuples pairing each sip_id with each date string
args = [(sip_id, date_str) for sip_id in all_us_prim_lids for date_str in date_list_str]

In [ ]:
size_list = []  # Batch size-bucketed analysis uses get_bbo() which is not Databricks Serverless-safe

In [ ]:
columns = [
    'mid_trade', 'bmll_trade_type', 'size_bucket',
    'size_sum_size_mo_1s', 'size_sumprod_size_mo_1s',
    'size_sum_size_mo_5s', 'size_sumprod_size_mo_5s',
    'size_sum_size_mo_10s', 'size_sumprod_size_mo_10s',
    'size_sum_size_mo_30s', 'size_sumprod_size_mo_30s',
    'size_sum_size_mo_60s', 'size_sumprod_size_mo_60s',
    'size_sum_size_mo_m1s', 'size_sumprod_size_mo_m1s',
    'size_sum_size_mo_m5s', 'size_sumprod_size_mo_m5s',
    'size_sum_size_mo_m10s', 'size_sumprod_size_mo_m10s',
    'size_sum_size_mo_m30s', 'size_sumprod_size_mo_m30s',
    'size_sum_size_mo_m60s', 'size_sumprod_size_mo_m60s'
]

dfs = []

for lst in size_list:
    # Ensure lst is a 2D list or array-like with length equal to len(columns)
    df = pd.DataFrame(lst, columns=columns)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=columns)
combined_df